# P74 — Inducción de árboles de decisión

## 1. Título y paper

**Paper:** *Induction of Decision Trees*  
**Autoría:** J. Ross Quinlan  
**Año y venue:** 1986 · Machine Learning, 1(1), 81–106  
**Nivel:** L2 · **Motor:** `id3`  
**Ficha completa:** [`P74_id3`](../../papers/foundational/P74_id3/README.md)

**Hito:** Aprende un modelo que una persona puede leer, eligiendo cada pregunta por cuánta incertidumbre elimina.

- [doi:10.1007/BF00116251](https://doi.org/10.1007/BF00116251)

> Este notebook implementa una **miniatura** del mecanismo. No reproduce el experimento original ni sus métricas: reproduce la idea para que se pueda inspeccionar y discutir.


## 2. Objetivos

1. Explicar qué problema resolvió el paper: Los clasificadores de la época eran cajas de números. En dominios donde alguien tiene que justificar la decisión, un modelo que no se puede leer no se puede usar.
2. Ejecutar una implementación mínima de la propuesta: Construir el árbol de arriba abajo eligiendo en cada nodo el atributo con mayor ganancia de información, y documentar el sesgo del criterio hacia los atributos con muchos valores.
3. Predecir el resultado antes de ejecutar, y contrastar la predicción con la salida.
4. Identificar al menos una limitación de la miniatura y una del paper original.
5. Conectar el hito con el siguiente eslabón de la ruta.


## 3. Prerrequisitos

- Python 3.11+ y el paquete del programa instalado (`pip install -e .`).
- Haber leído la guía [método de lectura en 5 pasadas](../../papers/guides/METODO_DE_LECTURA_EN_5_PASADAS.md).
- Hitos previos:
- P55
- Hunt et al. (1966), Concept Learning System


## 4. Intuición

En cada nodo, elegir la pregunta que más incertidumbre elimina. Se mide con la entropía: cuánto sabes antes de preguntar, cuánto sabes después. La diferencia es la ganancia. El problema es que preguntar «¿cuál es tu número de fila?» elimina TODA la incertidumbre.


## 5. Concepto mínimo

```text
Ganancia(S, A) = H(S) − Σ_v (|S_v|/|S|)·H(S_v)

Razón de ganancia = Ganancia / InfoDivisión,
    con InfoDivisión = −Σ_v (|S_v|/|S|)·log₂(|S_v|/|S|)

InfoDivisión crece con el número de valores → penaliza los atributos muy troceados
```


## 6. Código explicado

El motor aísla el mecanismo del paper con datos de juguete y salida inspeccionable.


In [ ]:
import json
import pathlib
import sys

ROOT = pathlib.Path.cwd()
while not (ROOT / "pyproject.toml").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
sys.path.insert(0, str(ROOT / "src"))

from ai_evolution.papers_lab import run_paper_lab


def show(value):
    print(json.dumps(value, ensure_ascii=False, indent=2))


In [ ]:
r = run_paper_lab('id3', seed=7)['result']
show(r)

## 7. Predicción antes de ejecutar

1. ¿Qué atributo elige la ganancia de información?
2. ¿Y si dejamos el identificador de fila dentro de la tabla?
3. ¿Lo arregla la razón de ganancia?

> Escribe tu respuesta aquí antes de continuar.


## 8. Experimento controlado

Se varía una sola cosa y se observa el efecto.


In [ ]:
for semilla in (1, 7, 42):
    r = run_paper_lab('id3', seed=semilla)
    print(f'semilla {semilla:>2} · evidencia principal:')
    for e in r['evidence']:
        print('   +', e)
    break  # determinista: basta una para ver la estructura
for semilla in (1, 7, 42):
    r = run_paper_lab('id3', seed=semilla)['result']
    print(f'semilla {semilla:>2} → claves: {list(r)[:4]}')

## 9. Salida interpretable

Con la tabla completa gana **«id»**, el identificador, con la ganancia máxima posible: separa perfectamente y no generaliza nada. El caso realista es «zona», con siete valores: gana en ganancia a «cielo» (0,3149 frente a 0,2467) y **pierde en razón de ganancia**. Ahí la corrección sí funciona; con el identificador dentro, no la salva ningún criterio.


## 10. Comentario pedagógico

Esa es la lección honesta y la que se cuenta mal en casi todos los cursos. La razón de ganancia corrige el sesgo hacia atributos muy troceados en el caso realista. No convierte un identificador en un atributo aceptable: eso es responsabilidad de quien prepara los datos, no del criterio de división.


## 11. Error o anti-patrón deliberado

Anti-patrón: dejar identificadores, fechas exactas o claves en la matriz de entrada.


In [ ]:
print('Un identificador separa perfectamente el entrenamiento y no generaliza nada.')
print('El arbol resultante tiene exactitud 100% dentro y la del azar fuera.')
print('Y lo mismo pasa con marcas de tiempo exactas o claves de cliente.')

## 12. Corrección

Lo que sí corrige el criterio, comprobado sobre atributos reales:


In [ ]:
r = run_paper_lab('id3', seed=7)['result']
for fila in r['tabla_de_ganancias']:
    print(f"{fila['atributo']:<12} valores={fila['valores']:>2} "
          f"ganancia={fila['ganancia']:<7} razon={fila['razon_de_ganancia']}")
print()
print('sin el identificador:', r['sin_el_identificador'])

## 13. Desafío guiado

Compara «zona» y «cielo» en las dos columnas y explica por qué la razón de ganancia invierte el orden.


In [ ]:
r = run_paper_lab('id3', seed=3)['result']
show(r)

## 14. Desafío autónomo

Construye el árbol completo sobre estos datos sin límite de profundidad, mide su exactitud dentro y fuera de muestra, y después pódalo. Documenta cuánto pierde dentro y cuánto gana fuera.


## 15. Evidencia de aprendizaje

Guarda la tabla de ganancias y razones de ganancia, y tu explicación de por qué un identificador no es un atributo.

Autoevaluación y respuestas esperadas: [ficha del paper](../../papers/foundational/P74_id3/README.md) · evaluación formal: [`assessments/papers/P74_id3.md`](../../assessments/papers/P74_id3.md)


## 16. Cierre

El árbol se lee, y sobreajusta. La pregunta siguiente es qué criterio usar cuando varios modelos aciertan lo mismo en entrenamiento.


## 17. Conexión con el siguiente hito

- P78
- P79

Ruta completa: [`papers/ROADMAP.md`](../../papers/ROADMAP.md)
